In [24]:
import json
import os
import urllib
from pathlib import Path
from IPython.display import clear_output

In [25]:
with open("trips_ids.txt", "r") as f:
        trips = set(line.strip() for line in f)

In [26]:
trips_folder = Path("trips_cache")
trips_folder.mkdir(exist_ok=True)

In [27]:
if Path("trips_cache/trips_cache_metadata").exists():
    with open("trips_cache/trips_cache_metadata", "r") as f:
        trips_cache_metadata = set(line.strip() for line in f)
else:
    trips_cache_metadata = set()

In [28]:
trips = trips.difference(trips_cache_metadata)

In [29]:
def register_trip(data):
    ut = data["ut"].capitalize().strip()
    line = str(data["cod"]).capitalize().strip()
    way = str(data["sentido"]).capitalize().strip()


    if not trips_folder.joinpath(ut).exists():
        trips_folder.joinpath(ut).mkdir(parents=True, exist_ok=True)
    if not trips_folder.joinpath(ut, line).exists():
        trips_folder.joinpath(ut, line).mkdir(parents=True, exist_ok=True)
    if not trips_folder.joinpath(ut, line, way).exists():
        trips_folder.joinpath(ut, line, way).mkdir(parents=True, exist_ok=True)

    with open(trips_folder.joinpath(ut, line, way, f"{data['servico']}.json"), "w") as f:
        json.dump(data, f, indent=4)

In [30]:
with open("trips_cache/trips_cache_metadata", "a") as f:
    for i, trip_id in enumerate(list(trips)):
        encoded_id = urllib.parse.quote(trip_id, safe='')
        url = f"https://paragens.amp.pt/acarto2/get_horarios_serv?id={encoded_id}"

        request = urllib.request.Request(url)
        with urllib.request.urlopen(request) as response:
            data = json.loads(response.read())
            register_trip(data)
            f.write(trip_id + "\n")
        if i % 100 == 0:
            clear_output(wait=True)
            print(f"Processed {i} out of {len(trips)} trips.")

Processed 8100 out of 8110 trips.
